In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from gensim.models import Doc2Vec
import umap
from spacy.lang.en import English
nlp = English(pipeline=[])
nlp.add_pipe("sentencizer")

/opt/miniconda3/envs/networks/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tweets_df = pd.read_csv("data/trump_tweets.csv")

tweets_df["date"] = pd.to_datetime(tweets_df["date"])
tweets_df["isPresident"] = (
    tweets_df["date"].between(pd.to_datetime("2017-1-20"), pd.to_datetime("2021-1-20"))
) | (
    tweets_df["date"] > pd.to_datetime("2025-1-20")
)  # YMD format

tweets_df = tweets_df.sort_values(by="date")

# ---------------------------------------------------------------

obama_tweets_df = pd.read_csv("data/obama.csv")
obama_tweets_df["date"] = pd.to_datetime(obama_tweets_df["Timestamp"]).dt.tz_localize(
    None
)
obama_tweets_df["isPresident"] = obama_tweets_df["date"].between(
    pd.to_datetime("2009-1-20"), pd.to_datetime("2017-1-20")
)  # YMD format

obama_tweets_df = obama_tweets_df.sort_values(by="date")
obama_tweets_df.drop("Emojis", axis=1, inplace=True)

# ---------------------------------------------------------------

vix_df = pd.read_csv("data/VIX_DAILY.csv")
vix_df["observation_date"] = pd.to_datetime(vix_df["observation_date"])
returns = np.array(np.log(vix_df["VIXCLS"])[1:-1]) - np.array(
    np.log(vix_df["VIXCLS"])[:-2]
)
vix_df["return"] = np.pad(returns, (0, 2), mode="constant")  # add 0's at the end
vix_df["absreturn"] = np.abs(vix_df["return"])

vix_df = vix_df.sort_values("observation_date")

merged = pd.merge_asof(
    tweets_df, vix_df, left_on="date", right_on="observation_date", direction="backward"
)
merged.dropna(inplace=True)
merged_bama = pd.merge_asof(
    obama_tweets_df,
    vix_df,
    left_on="date",
    right_on="observation_date",
    direction="backward",
)
merged_bama.dropna(inplace=True)

In [ ]:
merged_bama.head()
#list(merged_bama[merged_bama["isPresident"]]["Embedded_Text"])[:10]

,UserScreenName,UserName,Timestamp,Text,Embedded_text,Comments,Likes,Retweets,Image link,Tweet URL,date,isPresident,observation_date,VIXCLS,return,absreturn
263,Barack Obama,@BarackObama,2009-05-01T19:37:12.000Z,"Barack Obama\n@BarackObama\n·\nMay 1, 2009",The White House just joined Twitter. Follow \n...,2,4,115,[],https://twitter.com/BarackObama/status/1672136845,2009-05-01 19:37:12,True,2009-05-01,35.30,-0.022054,0.022054
271,Barack Obama,@BarackObama,2009-06-04T16:27:06.000Z,"Barack Obama\n@BarackObama\n·\nJun 4, 2009",“I've come here to Cairo to seek a new beginni...,2,12,276,[],https://twitter.com/BarackObama/status/2031182554,2009-06-04 16:27:06,True,2009-06-04,30.18,-0.018730,0.018730
276,Barack Obama,@BarackObama,2009-06-18T16:42:53.000Z,"Barack Obama\n@BarackObama\n·\nJun 18, 2009",RT \n@whitehouse\n WH Facebook page exclusive:...,1,2,172,[],https://twitter.com/BarackObama/status/2224610926,2009-06-18 16:42:53,True,2009-06-18,30.03,-0.070350,0.070350
279,Barack Obama,@BarackObama,2009-06-22T20:51:31.000Z,"Barack Obama\n@BarackObama\n·\nJun 22, 2009",Great news-- pharmaceuticals agree to reduce t...,19,22,96,[],https://twitter.com/BarackObama/status/2284416305,2009-06-22 20:51:31,True,2009-06-22,31.17,-0.019110,0.019110
310,Barack Obama,@BarackObama,2009-08-06T17:28:21.000Z,"Barack Obama\n@BarackObama\n·\nAug 6, 2009","In August, the health insurance reform debate ...",3,5,87,[],https://twitter.com/BarackObama/status/3166960437,2009-08-06 17:28:21,True,2009-08-06,25.67,-0.036094,0.036094


In [8]:
def tokenize(tweet):
    doc = nlp(str(tweet))
    tokens = []
    for sent in doc.sents:
        for token in sent:
            if not token.is_space:
                tokens.append(token.text.lower().strip())
        print(sent)
    return tokens

In [9]:
trump_tweets

['30332',
 'thank',
 'you',
 'for',
 'joining',
 'us',
 'at',
 'the',
 'lincoln',
 'memori',
 '...',
 '30333',
 'thank',
 'you',
 'for',
 'a',
 'wonderful',
 'evening',
 'in',
 'washingto',
 '...',
 '30334',
 'it',
 'all',
 'begins',
 'today',
 '!',
 'i',
 'will',
 'see',
 'you',
 'at',
 '11:00',
 'a',
 '...',
 '30335',
 'today',
 'we',
 'are',
 'not',
 'merely',
 'transferring',
 'power',
 'fro',
 '...',
 '30336',
 'power',
 'from',
 'washington',
 ',',
 'd.c.',
 'and',
 'giving',
 'it',
 'back',
 '...',
 '...',
 '44174',
 'rt',
 '@jackposobiec',
 ':',
 'breaking',
 ':',
 'us',
 'marines',
 'arrivin',
 '...',
 '44175',
 'get',
 'this',
 'straightened',
 'out',
 ',',
 'governor',
 '@gavinnews',
 '...',
 '44176',
 'wonderful',
 'account',
 'of',
 'u.s.',
 'embassy',
 '(',
 'iraq',
 ')',
 'vs.',
 't',
 '...',
 '44177',
 'how',
 'is',
 'the',
 'paris',
 'accord',
 'doing',
 '?',
 'do',
 'n’t',
 'ask',
 '!',
 'http',
 '...',
 '44178',
 'rt',
 '@whitehouse',
 ':',
 'americans',
 'saw',
 'pl

In [6]:
trump_tweets = tokenize(merged[merged["isPresident"]]["text"])
trump_space = Doc2Vec(trump_tweets, min_count=5, epochs=5)

obama_tweets = tokenize(merged_bama[merged_bama["isPresident"]]["Embedded_text"])
obama_space = Doc2Vec(obama_tweets, min_count=5, epochs=5)

30332    Thank you for joining us at the Lincoln Memori...
30333    Thank you for a wonderful evening in Washingto...
30334    It all begins today!
I will see you at 11:00 A...
30335    Today we are not merely transferring power fro...
30336    power from Washington, D.C. and giving it back...
                               ...                        
44174    RT @JackPosobiec: BREAKING: US Marines arrivin...
44175    Get this straightened out, Governor @GavinNews...
44176    Wonderful account of U.S. Embassy (Iraq) vs. t...
44177    How is the Paris Accord doing?
Don’t ask!
http...
44178    RT @WhiteHouse: Americans saw plenty of Washin...
Name: text, Length: 12895, dtype: str


AttributeError: 'str' object has no attribute 'words'

In [ ]:
reducer = umap.UMAP(n_components=2)
obama_small = reducer.fit_transform(obama_space)

In [10]:
from plotnine import ggplot, aes, geom_point

(ggplot(obama_small, aes(x="x", y="y")) + 
 geom_point())

NameError: name 'obama_small' is not defined